# 생성형 AI 기반 음성 모의면접 코치

수업에서 배운 **STT(귀) → LLM(머리) → TTS(입)** 구조를 면접 연습 서비스에 응용한 프로젝트입니다.

- STT: 사용자의 음성 답변을 텍스트로 변환
- LLM: 답변 점수, 장점, 개선점, 개선 답변, 후속 질문 생성
- TTS: 후속 질문을 면접관 목소리로 출력

> 실제 제출 전 `.env`에 Gemini API 키를 설정하고 본인 목소리로 실행 결과를 확인하세요.


## 1. 환경 설정

같은 폴더에 `.env` 파일과 `voice_utils.py`가 있어야 합니다.


In [ ]:
from pathlib import Path
from voice_utils import *

Path("recordings").mkdir(exist_ok=True)
print("환경 준비 완료")


## 2. 면접 질문과 설정


In [ ]:
QUESTIONS = {
    "자기소개": "1분 이내로 자기소개를 해주세요.",
    "지원동기": "우리 회사와 직무에 지원한 동기를 말씀해주세요.",
    "장단점": "본인의 장점과 단점을 말씀해주세요.",
    "협업 경험": "팀원과 협업하여 목표를 달성한 경험을 말씀해주세요.",
    "문제 해결": "어려운 문제를 해결했던 경험과 그 과정을 말씀해주세요.",
}

INTERVIEWER_VOICE = "Charon"
INTERVIEWER_STYLE = "professional Korean interviewer, calm, clear, neutral, moderate pace"

for name, q in QUESTIONS.items():
    print(f"[{name}] {q}")


## 3. 면접 답변 분석 프롬프트

답변에 없는 경험이나 숫자를 AI가 임의로 만들지 않도록 명시하고, 결과를 JSON으로 받습니다.


In [ ]:
INTERVIEW_PROMPT = """You are a professional Korean job interview coach for college students.

Interview question:
{question}

The applicant's answer was transcribed from speech:
{answer}

Evaluate only what is actually present in the applicant's answer. Do not invent experiences or facts.

Return JSON with exactly these keys:
  "score": integer from 0 to 100,
  "summary": one short Korean sentence summarizing the answer,
  "strengths": an array of 1 to 3 short Korean strengths,
  "improvements": an array of 1 to 3 concrete Korean improvement points,
  "star_feedback": one Korean sentence explaining whether Situation, Task, Action, Result are sufficiently concrete. If STAR is not appropriate for this question, say so briefly,
  "improved_answer": a concise improved Korean sample answer based only on the information the applicant actually provided. Do not fabricate company names, numbers, awards, or experiences,
  "followup_question": one natural Korean follow-up interview question based on the applicant's answer.

Evaluation criteria:
1. Relevance to the interview question
2. Specificity and evidence
3. Logical structure
4. Job-interview communication quality
5. Credibility: never reward invented or unsupported details

Keep the feedback constructive and practical for a college student.
"""

def evaluate_answer(question, answer):
    return llm_json(INTERVIEW_PROMPT.format(question=question, answer=answer))


## 4. 텍스트로 먼저 기능 확인


In [ ]:
sample_question = QUESTIONS["장단점"]
sample_answer = "저는 책임감이 강한 편이고 맡은 일은 끝까지 하려고 합니다. 단점은 일을 꼼꼼하게 하려다 시간이 오래 걸릴 때가 있다는 점입니다."

sample_result = evaluate_answer(sample_question, sample_answer)
for k, v in sample_result.items():
    print(f"{k}: {v}")


## 5. 음성 답변 → STT → AI 평가 → TTS


In [ ]:
def interview_turn(audio_path, question, voice=INTERVIEWER_VOICE):
    if not audio_path:
        raise ValueError("음성 파일이 필요합니다.")

    answer = stt(audio_path, lang="ko-KR", smart=True)
    result = evaluate_answer(question, answer)

    followup = result.get("followup_question", "추가로 말씀해 주실 내용이 있나요?")
    wav = tts(
        followup,
        voice=voice,
        style=INTERVIEWER_STYLE,
        filename="interviewer_followup.wav",
    )
    return answer, result, wav


## 6. 직접 녹음해서 실행

아래 셀은 40초 동안 답변을 녹음합니다.


In [ ]:
selected_question = QUESTIONS["자기소개"]
print("면접 질문:", selected_question)

answer_path = record("my_interview_answer.wav", seconds=40)
listen(answer_path)

transcript, result, followup_wav = interview_turn(answer_path, selected_question)
print("\n[STT 답변]")
print(transcript)
print("\n[AI 평가]")
for k, v in result.items():
    print(f"{k}: {v}")
print("\n[후속 질문 음성]")
listen(followup_wav, autoplay=True)


## 7. 결과 출력 함수


In [ ]:
def format_feedback(result):
    strengths = "\n".join(f"- {x}" for x in result.get("strengths", []))
    improvements = "\n".join(f"- {x}" for x in result.get("improvements", []))
    return f"""### 점수: {result.get('score', '-')} / 100

**요약**  
{result.get('summary', '')}

**잘한 점**  
{strengths or '-'}

**개선할 점**  
{improvements or '-'}

**STAR 관점 피드백**  
{result.get('star_feedback', '')}

**개선된 답변 예시**  
{result.get('improved_answer', '')}

**AI 면접관 후속 질문**  
{result.get('followup_question', '')}
"""


## 8. Gradio 웹 앱


In [ ]:
import gradio as gr

QUESTION_LABELS = list(QUESTIONS.keys())
VOICES = ["Charon", "Kore", "Puck", "Leda", "Sulafat", "Achird"]

def run_interview(audio_path, question_label, voice):
    if not audio_path:
        return "음성을 녹음하거나 업로드해 주세요.", "", None
    question = QUESTIONS[question_label]
    try:
        transcript, result, wav = interview_turn(audio_path, question, voice)
        return transcript, format_feedback(result), wav
    except Exception as e:
        return f"오류: {e}", "실행 중 오류가 발생했습니다. API 키와 호출 한도를 확인해 주세요.", None

with gr.Blocks(title="AI 음성 모의면접 코치") as app:
    gr.Markdown("# 🎤 AI 음성 모의면접 코치")
    gr.Markdown("면접 질문을 고르고 음성으로 답변하면 AI가 피드백과 후속 질문을 제공합니다.")

    with gr.Row():
        question_label = gr.Dropdown(QUESTION_LABELS, value="자기소개", label="면접 질문")
        voice = gr.Dropdown(VOICES, value=INTERVIEWER_VOICE, label="면접관 목소리")

    question_preview = gr.Textbox(value=QUESTIONS["자기소개"], label="선택된 질문", interactive=False)
    question_label.change(lambda x: QUESTIONS[x], question_label, question_preview)

    mic = gr.Audio(sources=["microphone", "upload"], type="filepath", label="내 답변")
    btn = gr.Button("AI 면접 피드백 받기", variant="primary")

    transcript_box = gr.Textbox(label="STT로 변환된 답변", lines=5)
    feedback_box = gr.Markdown()
    followup_audio = gr.Audio(label="AI 면접관 후속 질문", autoplay=True)

    btn.click(run_interview, [mic, question_label, voice], [transcript_box, feedback_box, followup_audio])

app.launch(inbrowser=True, inline=False)


## 9. 마무리

실제 제출 전 본인 음성으로 실행하고 결과가 정상 출력되는지 확인합니다.
